# 01 — Khám phá Dữ liệu (EDA) & Kiểm thử Tokenizer

**Mục tiêu notebook này:**
1. Thống kê cơ bản về bộ dữ liệu IWSLT'15 En–Vi
2. Vẽ phân phối độ dài câu
3. Kiểm thử BPE Tokenizer đã huấn luyện
4. Kiểm thử DataLoader và hàm tính BLEU

> **EDA** = Exploratory Data Analysis = khám phá, phân tích dữ liệu trước khi huấn luyện
> Giúp hiểu rõ dữ liệu: câu dài bao nhiêu? Từ nào phổ biến? Có bất thường không?

## 0. Cài đặt & Import

In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc dự án vào Python path để import src/
ROOT = Path.cwd().parent  # notebooks/ → dich-may-Anh-Viet/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

print(f"Root project: {ROOT}")

## 1. Thống kê dữ liệu gốc (data/raw)

In [ ]:
# Đọc dữ liệu
splits = {
    'train': ('data/raw/train.en',   'data/raw/train.vi'),
    'val'  : ('data/raw/tst2012.en', 'data/raw/tst2012.vi'),
    'test' : ('data/raw/tst2013.en', 'data/raw/tst2013.vi'),
}

print(f"{'Split':8s} | {'Số cặp câu':>12s} | {'Trung bình từ/câu EN':>22s} | {'Trung bình từ/câu VI':>22s}")
print('-' * 72)

for split, (en_path, vi_path) in splits.items():
    en_path = ROOT / en_path
    vi_path = ROOT / vi_path
    if not en_path.exists():
        print(f"{split:8s} | Chưa tải dữ liệu. Chạy: python scripts/download_data.py")
        continue

    en_lines = en_path.read_text(encoding='utf-8').splitlines()
    vi_lines = vi_path.read_text(encoding='utf-8').splitlines()

    en_lens = [len(s.split()) for s in en_lines]
    vi_lens = [len(s.split()) for s in vi_lines]

    print(f"{split:8s} | {len(en_lines):>12,} | {np.mean(en_lens):>22.1f} | {np.mean(vi_lens):>22.1f}")

## 2. Phân phối độ dài câu (histogram)

Xem câu dài bao nhiêu từ là phổ biến nhất → chọn `max_len` phù hợp.

In [ ]:
train_en_path = ROOT / 'data/processed/train.en'
train_vi_path = ROOT / 'data/processed/train.vi'

if train_en_path.exists():
    en_lines = [l.rstrip('\r\n') for l in train_en_path.open(encoding='utf-8')]
    vi_lines = [l.rstrip('\r\n') for l in train_vi_path.open(encoding='utf-8')]

    en_lens = [len(s.split()) for s in en_lines]
    vi_lens = [len(s.split()) for s in vi_lines]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(en_lens, bins=50, color='steelblue', edgecolor='white')
    axes[0].set_title('Phân phối độ dài câu Tiếng Anh (train đã làm sạch)')
    axes[0].set_xlabel('Số từ / câu')
    axes[0].set_ylabel('Số câu')
    axes[0].axvline(128, color='red', linestyle='--', label='max_words=128')
    axes[0].legend()

    axes[1].hist(vi_lens, bins=50, color='coral', edgecolor='white')
    axes[1].set_title('Phân phối độ dài câu Tiếng Việt (train đã làm sạch)')
    axes[1].set_xlabel('Số từ / câu')
    axes[1].axvline(128, color='red', linestyle='--', label='max_words=128')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(ROOT / 'notebooks/sentence_length_dist.png', dpi=100)
    plt.show()
    print(f"EN: max={max(en_lens)}, median={np.median(en_lens):.0f}, p95={np.percentile(en_lens,95):.0f}, p99={np.percentile(en_lens,99):.0f}")
    print(f"VI: max={max(vi_lens)}, median={np.median(vi_lens):.0f}, p95={np.percentile(vi_lens,95):.0f}, p99={np.percentile(vi_lens,99):.0f}")
else:
    print("Chưa có dữ liệu. Chạy: python scripts/download_data.py")

## 3. Xem vài cặp câu mẫu

In [ ]:
if train_en_path.exists():
    print("Vài cặp câu mẫu từ tập train:\n")
    for i in range(5):
        print(f"[{i+1}] EN: {en_lines[i]}")
        print(f"    VI: {vi_lines[i]}")
        print()

## 4. Kiểm thử BPE Tokenizer

In [ ]:
en_tok_path = ROOT / 'data/tokenizer/bpe_en.json'
vi_tok_path = ROOT / 'data/tokenizer/bpe_vi.json'

if en_tok_path.exists() and vi_tok_path.exists():
    from src.data.tokenizer import BPETokenizer

    src_tok = BPETokenizer(en_tok_path)
    tgt_tok = BPETokenizer(vi_tok_path)
    print(f"EN vocab_size={src_tok.vocab_size}, VI vocab_size={tgt_tok.vocab_size}")
    print(f"PAD={src_tok.pad_id}, UNK={src_tok.unk_id}, BOS={src_tok.bos_id}, EOS={src_tok.eos_id}\n")

    test_pairs = [
        (src_tok, "I'd like \"AI: 100% / $50;\" in 2025!"),
        (tgt_tok, "Tôi yêu Việt Nam (100%)!"),
    ]

    for tok, sent in test_pairs:
        tokens = tok.encode_tokens(sent)
        ids = tok.encode(sent)
        decoded = tok.decode(ids)
        print(f"Câu gốc : {sent}")
        print(f"Tokens  : {tokens}")
        print(f"IDs     : {ids}")
        print(f"Giải mã : {decoded}")
        print()
else:
    print("Chưa có tokenizer. Chạy: python src/data/tokenizer.py")

## 5. Kiểm thử DataLoader

In [ ]:
processed_en = ROOT / 'data/processed/train.en'

if en_tok_path.exists() and vi_tok_path.exists() and processed_en.exists():
    from src.data.dataset import create_dataloader

    loader = create_dataloader(
        en_path=ROOT / 'data/processed/train.en',
        vi_path=ROOT / 'data/processed/train.vi',
        src_tokenizer=src_tok,
        tgt_tokenizer=tgt_tok,
        batch_size=4,
        shuffle=False,
        max_len=128,
    )

    batch = next(iter(loader))
    for k, v in batch.items():
        print(f"{k:15s}: shape={tuple(v.shape)}, dtype={v.dtype}")
    print(f"\nGiải mã src[0]: {src_tok.decode(batch['src_ids'][0])}")
    print(f"Giải mã tgt[0]: {tgt_tok.decode(batch['tgt_output_ids'][0])}")
    print("\nDataLoader OK!")
else:
    print("Cần có tokenizer và dữ liệu đã xử lý. Xem README.md phần 2.")

## 6. Kiểm thử hàm tính BLEU

In [ ]:
from src.evaluation.metrics import compute_bleu, compute_sentence_bleu

# Test 1: câu giống hệt nhau → BLEU phải ~100
score_perfect = compute_bleu(["tôi yêu bạn"], ["tôi yêu bạn"])
print(f"Câu giống hệt → BLEU = {score_perfect:.2f}  (kỳ vọng ~100)")

# Test 2: gần giống → BLEU trung bình
score_close = compute_bleu(["tôi yêu bạn"], ["tôi yêu em"])
print(f"Gần giống     → BLEU = {score_close:.2f}")

# Test 3: khác hẳn → BLEU gần 0
score_bad = compute_bleu(["tôi yêu bạn"], ["xin chào thế giới"])
print(f"Khác hẳn      → BLEU = {score_bad:.2f}  (kỳ vọng ~0)")

print("\nBLEU function OK!")